<a href="https://colab.research.google.com/github/ihinck413/Econ-5120/blob/main/5200_HW5_Problem_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 3: Four valuation methods for MSFT

Intrinsic value of Microsoft with the four DCF methods from the lecture: FCFF at the after-tax WACC, FCFE at the cost of equity, CCF at the pre-tax WACC, and APV.

**Data** (in `./data`)
- `msft_companyfacts.json`: SEC EDGAR XBRL company facts (10-K FY2026, 10-Q Q1 FY2027 to 2026-09-30)
- `msft_nasdaq.json`, `msft_divs.json`: Nasdaq daily closes (2016-11-07 to 2026-11-06) and dividend history
- `F-F_Research_Data_Factors.csv`: Ken French monthly factors (to 2026-09)
- `dgs10.csv`: FRED 10-year Treasury yield

**Notation** : V = equity, B = debt, X = FCFF, D = FCFE, CCF = X + τrB, R^E = cost of equity, R^W = after-tax WACC, R^V = pre-tax WACC, r = risk-free rate = cost of debt.



In [1]:
from datetime import date
from pathlib import Path
import json

import numpy as np
import pandas as pd

HERE = Path.cwd()
DATA, OUT = HERE / "data", HERE / "output"
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160)
bn = lambda v: f"{v:,.1f}"

## Inputs from the filings

The base year is the trailing 12 months (TTM) to Sep 2026: FY2026 + Q1 FY27 − Q1 FY26. τ is the TTM effective tax rate.

In [5]:
import os

# Fallback to current working directory if DATA directory is empty or does not exist
DATA_DIR = DATA if (DATA.exists() and any(DATA.iterdir())) else HERE

G = json.load(open(DATA_DIR / "msft_companyfacts.json"))["facts"]["us-gaap"]


def flow(tag, start, end):
    """Duration fact (USD bn) for an exact start/end."""
    for x in G[tag]["units"]["USD"]:
        if x.get("start") == start and x["end"] == end:
            return x["val"] / 1e9
    raise KeyError((tag, start, end))


def stock(tag, end):
    for x in G[tag]["units"]["USD"]:
        if "start" not in x and x["end"] == end:
            return x["val"] / 1e9
    raise KeyError((tag, end))


def ttm(tag):
    """Trailing 12 months to 2026-09-30 = FY2026 + Q1 FY27 - Q1 FY26."""
    return (flow(tag, "2025-07-01", "2026-06-30") + flow(tag, "2026-07-01", "2026-09-30")
            - flow(tag, "2025-07-01", "2025-09-30"))


def fy(tag, y):
    return flow(tag, f"{y-1}-07-01", f"{y}-06-30")


REV, EBIT = "RevenueFromContractWithCustomerExcludingAssessedTax", "OperatingIncomeLoss"
PRETAX = "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"
base = dict(
    revenue=ttm(REV), ebit=ttm(EBIT),
    da=ttm("Depreciation") + ttm("AmortizationOfIntangibleAssets"),
    capex=ttm("PaymentsToAcquirePropertyPlantAndEquipment"),
    tax=ttm("IncomeTaxExpenseBenefit"), pretax=ttm(PRETAX),
    interest=ttm("InterestExpenseNonoperating"),
)
tau = base["tax"] / base["pretax"]

print("Base year (TTM to 2026-09-30), $bn")
for k, v in base.items():
    print(f"  {k:9s} {v:8.2f}")
print(f"  tax rate {tau:.3%}")

Base year (TTM to 2026-09-30), $bn
  revenue     346.44
  ebit        163.19
  da           42.50
  capex       135.92
  tax          34.95
  pretax      177.59
  interest      3.22
  tax rate 19.683%


### History by fiscal year

FCFF = EBIT(1−τ) + D&A − capex − ΔNWC. Revenue and EBIT have nearly doubled since FY22, but FCFF has fallen: the AI buildout has pushed capex from 12% to 35% of revenue.

In [6]:
# Historical FCFF by fiscal year (for the growth discussion)
WC_TAGS = {  # sign: +1 means a positive XBRL value is a cash outflow (asset increase)
    "IncreaseDecreaseInAccountsReceivable": 1, "IncreaseDecreaseInInventories": 1,
    "IncreaseDecreaseInOtherCurrentAssets": 1, "IncreaseDecreaseInAccountsPayable": -1,
    "IncreaseDecreaseInContractWithCustomerLiability": -1,
    "IncreaseDecreaseInOtherCurrentLiabilities": -1, "IncreaseDecreaseInAccruedIncomeTaxesPayable": -1,
}
hist = []
for y in range(2022, 2027):
    t_y = fy("IncomeTaxExpenseBenefit", y) / fy(PRETAX, y)
    dnwc = sum(s * fy(t, y) for t, s in WC_TAGS.items())
    row = dict(FY=y, revenue=fy(REV, y), ebit=fy(EBIT, y), tax_rate=t_y,
               da=fy("Depreciation", y) + fy("AmortizationOfIntangibleAssets", y),
               capex=fy("PaymentsToAcquirePropertyPlantAndEquipment", y), dnwc=dnwc)
    row["fcff"] = row["ebit"] * (1 - t_y) + row["da"] - row["capex"] - dnwc
    hist.append(row)
hist = pd.DataFrame(hist).set_index("FY")
hist["growth"] = hist.revenue.pct_change()
hist["ebit_margin"] = hist.ebit / hist.revenue
hist["capex_pct"] = hist.capex / hist.revenue
hist["da_pct"] = hist.da / hist.revenue
hist["dnwc_per_drev"] = hist.dnwc / hist.revenue.diff()

hist.round(3)

,revenue,ebit,tax_rate,da,capex,dnwc,fcff,growth,ebit_margin,capex_pct,da_pct,dnwc_per_drev
FY,,,,,,,,,,,,
2022,198.270,83.383,0.131,14.6,23.886,-2.426,65.589,NaN,0.421,0.120,0.074,NaN
2023,211.915,88.523,0.190,13.5,28.107,0.108,57.008,0.069,0.418,0.133,0.064,0.008
2024,245.122,109.433,0.182,20.0,44.477,-7.892,72.897,0.157,0.446,0.181,0.082,-0.238
2025,281.724,128.528,0.176,28.0,64.551,1.425,67.893,0.149,0.456,0.229,0.099,0.039
2026,331.839,155.237,0.194,39.0,115.948,-3.776,51.955,0.178,0.468,0.349,0.118,-0.075


### Balance sheet at 2026-09-30

Finance leases count as debt, because they're data-center financing and their interest is in interest expense. Cash, short-term investments and long-term investments (including the OpenAI equity-method stake) are added back as non-operating assets.

In [8]:
# Balance sheet at 2026-09-30
BS = "2026-09-30"
debt_fin = stock("LongTermDebt", BS)                 # bonds, current + noncurrent
fin_lease = stock("FinanceLeaseLiability", BS)       # treated as debt
B0 = debt_fin + fin_lease
cash = stock("CashCashEquivalentsAndShortTermInvestments", BS)
lt_inv = stock("LongTermInvestments", BS)            # incl. equity-method (OpenAI) stake
nonop = cash + lt_inv

dei = json.load(open(DATA_DIR / "msft_companyfacts.json"))["facts"]["dei"]
shares = max(dei["EntityCommonStockSharesOutstanding"]["units"]["shares"], key=lambda x: x["end"])["val"] / 1e9

print(f"B = {bn(debt_fin)} bonds + {bn(fin_lease)} finance leases = {bn(B0)}")
print(f"cash + ST investments {bn(cash)}, LT investments {bn(lt_inv)}, shares {shares:.4f}bn")

B = 35.5 bonds + 71.0 finance leases = 106.5
cash + ST investments 64.5, LT investments 37.8, shares 7.4262bn


## (a) FCFF forecast

**Assumptions**
- **Growth:** 17% in year 1, in line with FY26 (+17.8%) and Q1 FY27 (+18.8%). It then fades to 8% by year 5. Terminal g = 4%, roughly nominal GDP growth and below r.
- **EBIT margin:** flat at 46%. Operating leverage and rising depreciation from the capex roughly cancel.
- **Capex:** fades from 39% to 19% of revenue as the buildout matures. D&A rises from 13% to 16%.
- **ΔNWC:** −3% of Δrevenue, because unearned revenue funds working capital. The FY23–26 average was about −7%.

The terminal value is Gordon's formula: TV₅ = X₅(1+g)/(R^W − g).

In [9]:
ASSUME = pd.DataFrame({
    "growth":   [0.17, 0.15, 0.12, 0.10, 0.08],
    "ebit_margin": [0.46] * 5,
    "da_pct":   [0.13, 0.14, 0.15, 0.155, 0.16],
    "capex_pct": [0.36, 0.32, 0.27, 0.22, 0.19],
}, index=pd.Index(range(1, 6), name="t"))
DNWC_PER_DREV = -0.03          # deferred revenue funds working capital (see history)
G_TERM = 0.04

fc = ASSUME.copy()
rev_prev = base["revenue"]
revs = []
for g in fc.growth:
    rev_prev *= 1 + g
    revs.append(rev_prev)
fc["revenue"] = revs
fc["ebit"] = fc.revenue * fc.ebit_margin
fc["nopat"] = fc.ebit * (1 - tau)
fc["da"] = fc.revenue * fc.da_pct
fc["capex"] = fc.revenue * fc.capex_pct
fc["dnwc"] = DNWC_PER_DREV * fc.revenue.diff().fillna(fc.revenue.iloc[0] - base["revenue"])
fc["X"] = fc.nopat + fc.da - fc.capex - fc.dnwc
base_X = base["ebit"] * (1 - tau) + base["da"] - base["capex"]  # TTM, ignoring dNWC

print(f"TTM FCFF (base year) = {base_X:.1f}")
fc.round(3)

TTM FCFF (base year) = 37.6


,growth,ebit_margin,da_pct,capex_pct,revenue,ebit,nopat,da,capex,dnwc,X
t,,,,,,,,,,,
1,0.17,0.46,0.130,0.36,405.341,186.457,149.756,52.694,145.923,-1.767,58.294
2,0.15,0.46,0.140,0.32,466.142,214.425,172.219,65.260,149.165,-1.824,90.138
3,0.12,0.46,0.150,0.27,522.079,240.156,192.885,78.312,140.961,-1.678,131.914
4,0.10,0.46,0.155,0.22,574.287,264.172,212.174,89.014,126.343,-1.566,176.411
5,0.08,0.46,0.160,0.19,620.230,285.306,229.148,99.237,117.844,-1.378,211.919


## (b) CAPM and WACC

β comes from regressing monthly MSFT total excess returns on Mkt-RF. The 60-month estimate is the base case. The equity premium is 5%, a conventional middle value below the 1926–2026 historical average of Mkt-RF. Weights use market values.

In [12]:
rows = json.load(open(DATA_DIR / "msft_nasdaq.json"))["data"]["tradesTable"]["rows"]
px = pd.Series({pd.Timestamp(date(int(r["date"][6:]), int(r["date"][:2]), int(r["date"][3:5]))):
                float(r["close"].strip("$").replace(",", "")) for r in rows}).sort_index()
price, price_date = px.iloc[-1], px.index[-1].date()
dv = json.load(open(DATA_DIR / "msft_divs.json"))["data"]["dividends"]["rows"]
divs = pd.Series({pd.to_datetime(r["exOrEffDate"]): float(r["amount"].strip("$")) for r in dv
                  if r["type"] == "Cash" and r["exOrEffDate"] != "N/A"}).sort_index()

pm = px.resample("ME").last()
dm = divs.resample("ME").sum().reindex(pm.index, fill_value=0.0)
ret = (pm + dm) / pm.shift(1) - 1
ret.index = ret.index.to_period("M")

ff = pd.read_csv(DATA_DIR / "F-F_Research_Data_Factors.csv", skiprows=4, index_col=0)
ff = ff[ff.index.astype(str).str.strip().str.fullmatch(r"\d{6}")].astype(float) / 100
ff.index = pd.PeriodIndex(ff.index.astype(str).str.strip(), freq="M")


def capm(start, end):
    d = pd.concat([ret.rename("r"), ff], axis=1).loc[start:end].dropna()
    y, x = d.r - d.RF, d["Mkt-RF"]
    X = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    s2 = resid @ resid / (len(y) - 2)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    r2 = 1 - resid.var() / y.var()
    return dict(window=f"{d.index[0]}..{d.index[-1]}", n=len(d), alpha_m=coef[0], beta=coef[1],
                se_beta=se[1], r2=r2)


beta_tbl = pd.DataFrame([capm("2021-10", "2026-09"), capm("2016-12", "2026-09")])
beta = beta_tbl.beta.iloc[0]                     # 60-month estimate is the base case
ff_erp_hist = ff["Mkt-RF"].mean() * 12           # 1926-2026 arithmetic mean, for reference

dgs = pd.read_csv(DATA_DIR / "dgs10.csv", index_col=0, na_values=".").dropna().iloc[:, 0]
RF = dgs.iloc[-1] / 100
ERP = 0.05

E_mkt = price * shares
L = B0 / (E_mkt + B0)                            # market-value debt weight


def rates(r, beta=beta, erp=ERP):
    RE = r + beta * erp
    RV = (1 - L) * RE + L * r                    # pre-tax WACC
    RW = (1 - L) * RE + L * r * (1 - tau)        # after-tax WACC
    return RE, RW, RV


RE, RW, RV = rates(RF)

print(beta_tbl.round(4).to_string(index=False))
print(f"\nr (10y UST, {dgs.index[-1]}) = {RF:.2%};  ERP = {ERP:.1%}  (FF 1926-2026 mean Mkt-RF = {ff_erp_hist:.2%})")
print(f"price {price:.2f} on {price_date}; E = {bn(E_mkt)}; B = {bn(B0)}; L = B/(E+B) = {L:.3%}")
print(f"R^E = {RE:.3%}   pre-tax WACC R^V = {RV:.3%}   after-tax WACC R^W = {RW:.3%}")

          window   n  alpha_m   beta  se_beta     r2
2021-10..2026-09  60   0.0024 1.0400   0.1785 0.3691
2016-12..2026-09 118   0.0095 0.9169   0.1027 0.4075

r (10y UST, 2026-11-06) = 4.86%;  ERP = 5.0%  (FF 1926-2026 mean Mkt-RF = 8.33%)
price 530.61 on 2026-11-06; E = 3,940.4; B = 106.5; L = B/(E+B) = 2.631%
R^E = 10.060%   pre-tax WACC R^V = 9.923%   after-tax WACC R^W = 9.898%


## (c) Value by the four methods

Each method is the slide's recursion, solved backward from year 5.
- **FCFF, FCFE and CCF.** A constant WACC implies debt is rebalanced to B_t = L(V_t + B_t). With that debt path, the three recursions are algebraically identical.
- **APV with the tax shield discounted at r.** This treats the shield as riskless, which holds only if today's debt is held fixed forever. Then PV(TS) = τB, and R^U comes from unlevering under that same policy.
- **APV (iv′).** Uses the rebalanced-debt policy, with the shield discounted at R^U = R^V. It matches the other three.

In [13]:
def value_all(r=RF, g_term=G_TERM, beta=beta, erp=ERP):
    RE, RW, RV = rates(r, beta, erp)
    X = fc.X.values
    T = len(X)
    # -- FCFF at after-tax WACC: V+B = (X' + V'+B')/(1+R^W); terminal: Gordon at t=T
    F = np.zeros(T + 1)
    F[T] = X[-1] * (1 + g_term) / (RW - g_term)
    for t in range(T - 1, -1, -1):
        F[t] = (X[t] + F[t + 1]) / (1 + RW)
    tv_share = F[T] / (1 + RW) ** T / F[0]
    # Debt policy behind a constant WACC: B_t = L (V_t + B_t)
    Bp = L * F
    # -- FCFE at cost of equity: V = (D' + V')/(1+R^E), D' = X' - (1-tau) r B + (B' - B)
    VE = np.zeros(T + 1)
    VE[T] = F[T] - Bp[T]          # beyond T everything grows at g, so V_T = (1-L) F_T
    for t in range(T - 1, -1, -1):
        D = X[t] - (1 - tau) * r * Bp[t] + (Bp[t + 1] - Bp[t])
        VE[t] = (D + VE[t + 1]) / (1 + RE)
    # -- CCF at pre-tax WACC: V+B = (CCF' + V'+B')/(1+R^V), CCF' = X' + tau r B
    C = np.zeros(T + 1)
    C[T] = (X[-1] * (1 + g_term) + tau * r * Bp[T]) / (RV - g_term)
    for t in range(T - 1, -1, -1):
        C[t] = (X[t] + tau * r * Bp[t] + C[t + 1]) / (1 + RV)
    # -- APV with tax shields discounted at r: assumes the actual debt B0 is held fixed forever
    #    (riskless shields); unlever R^E consistently with that policy (Hamada/MM):
    #    R^E = R^U + (R^U - r)(1-tau) B/V  =>  R^U = (V R^E + (1-tau) B r) / (V + (1-tau) B)
    E0, B0_ = E_mkt, B0
    RU = (E0 * RE + (1 - tau) * B0_ * r) / (E0 + (1 - tau) * B0_)
    VU = np.zeros(T + 1)
    VU[T] = X[-1] * (1 + g_term) / (RU - g_term)
    for t in range(T - 1, -1, -1):
        VU[t] = (X[t] + VU[t + 1]) / (1 + RU)
    TS = tau * B0_                               # PV of tau*r*B0 forever at r
    # APV variant with growing debt (B_t = L F_t) and shields at R^U = R^V (Harris-Pringle)
    VU_hp = np.zeros(T + 1)
    VU_hp[T] = X[-1] * (1 + g_term) / (RV - g_term)
    TS_hp = np.zeros(T + 1)
    TS_hp[T] = tau * r * Bp[T] / (RV - g_term)
    for t in range(T - 1, -1, -1):
        VU_hp[t] = (X[t] + VU_hp[t + 1]) / (1 + RV)
        TS_hp[t] = (tau * r * Bp[t] + TS_hp[t + 1]) / (1 + RV)

    # Equity to today's shareholders. Firm-value methods: (V+B) - actual debt + non-op assets.
    # FCFE: V from the recursion assumes debt Bp[0]; move to actual debt B0 one-for-one.
    eq = {
        "(i) FCFF @ after-tax WACC": F[0] - B0 + nonop,
        "(ii) FCFE @ cost of equity": VE[0] - (B0 - Bp[0]) + nonop,
        "(iii) CCF @ pre-tax WACC": C[0] - B0 + nonop,
        "(iv) APV, TS @ r (fixed debt)": VU[0] + TS - B0 + nonop,
        "(iv') APV, TS @ R^U (rebalanced debt)": VU_hp[0] + TS_hp[0] - B0 + nonop,
    }
    detail = dict(RE=RE, RW=RW, RV=RV, RU=RU, firm=F[0], tv_share=tv_share, VU=VU[0], TS=TS,
                  VU_hp=VU_hp[0], TS_hp=TS_hp[0], B_model0=Bp[0], VE0=VE[0])
    return eq, detail


eq, det = value_all()

tab = pd.DataFrame({"equity_$bn": eq}).assign(per_share=lambda d: d["equity_$bn"] / shares)
tab["vs_price"] = tab.per_share / price - 1
print(f"market price {price:.2f};  TV share of firm value {det['tv_share']:.1%};  "
      f"R^U (fixed-debt unlevering) = {det['RU']:.3%}")
tab.round(3)

market price 530.61;  TV share of firm value 82.9%;  R^U (fixed-debt unlevering) = 9.950%


,equity_$bn,per_share,vs_price
(i) FCFF @ after-tax WACC,2807.088,377.998,-0.288
(ii) FCFE @ cost of equity,2807.088,377.998,-0.288
(iii) CCF @ pre-tax WACC,2807.088,377.998,-0.288
"(iv) APV, TS @ r (fixed debt)",2801.679,377.270,-0.289
"(iv') APV, TS @ R^U (rebalanced debt)",2807.088,377.998,-0.288


### What the market price implies

Solve for the terminal g that makes the FCFF value equal the market price, holding the base-case rates fixed.

In [14]:
# Market-implied terminal growth at base rates (bisection on the FCFF value)
lo, hi = 0.0, RW - 1e-4
for _ in range(100):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if value_all(RF, mid)[0]["(i) FCFF @ after-tax WACC"] / shares < price else (lo, mid)
g_implied = mid
print(f"market-implied terminal g at base rates: {g_implied:.2%}")

market-implied terminal g at base rates: 5.86%


## (d) Sensitivity to r and g

Changing r moves R^E, R^W and R^V one-for-one; β, the equity premium and the FCFF forecasts are held fixed. The first table is value per share from the FCFF method. The second is the share of firm value that comes from the terminal value.

In [15]:
r_grid = [0.05, 0.06, 0.07, 0.08, 0.09, 0.10]
g_grid = [0.03, 0.04, 0.05, 0.06]
sens = pd.DataFrame(index=[f"r={r:.0%}" for r in r_grid], columns=[f"g={g:.0%}" for g in g_grid], dtype=float)
tvsh = sens.copy()
for r in r_grid:
    for g in g_grid:
        e, d = value_all(r, g)
        sens.loc[f"r={r:.0%}", f"g={g:.0%}"] = e["(i) FCFF @ after-tax WACC"] / shares
        tvsh.loc[f"r={r:.0%}", f"g={g:.0%}"] = d["tv_share"]

print("Value per share (FCFF method)")
print(sens.round(0).to_string())
print("\nShare of firm value from the terminal value")
print(tvsh.round(3).to_string())

Value per share (FCFF method)
        g=3%   g=4%   g=5%   g=6%
r=5%   323.0  369.0  433.0  528.0
r=6%   279.0  312.0  356.0  418.0
r=7%   245.0  270.0  302.0  344.0
r=8%   217.0  237.0  261.0  292.0
r=9%   195.0  210.0  229.0  252.0
r=10%  176.0  189.0  204.0  222.0

Share of firm value from the terminal value
        g=3%   g=4%   g=5%   g=6%
r=5%   0.801  0.826  0.851  0.878
r=6%   0.776  0.800  0.825  0.851
r=7%   0.753  0.776  0.800  0.824
r=8%   0.730  0.752  0.775  0.799
r=9%   0.708  0.729  0.751  0.774
r=10%  0.687  0.707  0.728  0.751
